# LLM-ассистент для анализа резюме (HR-ассистент)

**Прикладная область:** HR-технологии — извлечение структурированной информации из текстов резюме кандидатов.

**Входные данные:** Тексты резюме в свободной форме (имя, возраст, должность, навыки, опыт, образование, достижения).

**Выходные данные (JSON):**
| Поле | Тип | Описание |
|------|-----|----------|
| `candidate_name` | str | ФИО кандидата |
| `desired_position` | str | Желаемая должность |
| `skills` | list[str] | Ключевые навыки |
| `experience_years` | float | Общий опыт (лет) |
| `education` | str | Образование |
| `top_achievement` | str | Главное достижение |
| `suitability_score` | float | Оценка релевантности (0–10) |
| `comment` | str | Краткий комментарий |

**Почему OpenRouter вместо Groq:** Groq API заблокирован на территории РФ. OpenRouter — OpenAI-совместимый API, предоставляющий бесплатные модели.

## 1. Установка и настройка клиента

Устанавливаем библиотеки. Ключ хранится в `.env` и не захардкожен в коде.

In [1]:
import sys
!{sys.executable} -m pip install "openai>=1.0,<2.0" python-dotenv groq --quiet



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import time
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from collections import Counter, defaultdict

# Загружаем .env из той же папки, где лежит ноутбук
env_path = Path(".") / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("API-ключ не найден! Добавьте OPENROUTER_API_KEY=sk-or-... в .env")

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)

print("Клиент OpenRouter инициализирован. Ключ:", api_key[:10] + "...")


Клиент OpenRouter инициализирован. Ключ: sk-or-v1-6...


## 2. Демонстрация проблемы неструктурированного вывода и решение через JSON-режим

### 2.1 Без JSON-режима

Модель возвращает произвольный текст, который сложно парсить автоматически.

In [3]:
# Без JSON-режима
resp = client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[
        {"role": "system", "content": "Извлеки информацию из резюме: имя, навыки, опыт."},
        {"role": "user", "content": "Иван Петров, Java-разработчик, 5 лет. Java, Spring, PostgreSQL."}
    ],
)
print("Ответ без JSON-режима:")
print(resp.choices[0].message.content)


Ответ без JSON-режима:
**Имя:** Иван Петров  
**Навыки:** Java, Spring, PostgreSQL  
**Опыт:** Java‑разработчик, 5 лет


### 2.2 С JSON-режимом

Параметр `response_format={"type": "json_object"}` заставляет модель возвращать только валидный JSON.

In [4]:
# С JSON-режимом
resp = client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[
        {"role": "system", "content": "Ты — HR-ассистент. Извлеки данные из резюме и верни строгий JSON."},
        {"role": "user", "content": "Иван Петров, Java-разработчик, 5 лет. Java, Spring, PostgreSQL. МГУ."}
    ],
    response_format={"type": "json_object"},
)
parsed = json.loads(resp.choices[0].message.content)
print(json.dumps(parsed, ensure_ascii=False, indent=2))
print(f"\nТокенов: {resp.usage.total_tokens}")


{
  "name": "Иван Петров",
  "position": "Java-разработчик",
  "years_experience": 5,
  "skills": [
    "Java",
    "Spring",
    "PostgreSQL"
  ],
  "education": "МГУ"
}

Токенов: 276


## 3. Основная функция обработки

Функция принимает текст резюме, отправляет в LLM, парсит JSON. Реализованы:
- системный промпт с описанием полей JSON
- `response_format` для гарантии JSON
- `json.loads()` для парсинга
- `try/except` для обработки ошибок (сетевые сбои, невалидный JSON)
- повторные попытки при временных ошибках

In [5]:
def process_resume(text, model="nvidia/nemotron-3-super-120b-a12b:free", max_retries=3):
    system_prompt = (
        "Ты — HR-ассистент. Извлеки данные из резюме кандидата.\n"
        "Верни ТОЛЬКО JSON без дополнительного текста.\n"
        "Поля JSON:\n"
        "- candidate_name (str): полное имя кандидата\n"
        "- desired_position (str): желаемая должность\n"
        "- skills (list[str]): список ключевых навыков\n"
        "- experience_years (float): общий опыт работы в годах\n"
        "- education (str): образование\n"
        "- top_achievement (str): главное достижение\n"
        "- suitability_score (float): оценка от 0 до 10\n"
        "- comment (str): краткий комментарий"
    )
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": text}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            raw = resp.choices[0].message.content
            data = json.loads(raw)
            data["_model"] = model.split("/")[-1].split(":")[0]
            data["_tokens"] = resp.usage.total_tokens if resp.usage else 0
            data["_time"] = time.time()
            return data
        except Exception as e:
            if attempt == max_retries:
                return {
                    "candidate_name": f"ERROR: {str(e)[:80]}",
                    "desired_position": "",
                    "skills": [],
                    "experience_years": 0,
                    "education": "",
                    "top_achievement": "",
                    "suitability_score": 0,
                    "comment": str(e)[:200],
                    "_model": model.split("/")[-1].split(":")[0],
                    "_tokens": 0,
                    "_time": time.time(),
                    "_error": True,
                }
            time.sleep(2 ** attempt)  # exponential backoff

print("Функция process_resume определена")


Функция process_resume определена


## 4. Пайплайн обработки 6 резюме

Подготовим 6 тестовых резюме разных специальностей и пропустим через функцию.

In [6]:
resumes = [
    "Иван Петров, 30 лет. Java-разработчик, 5 лет опыта. Навыки: Java, Spring, PostgreSQL, Docker, Kubernetes. Образование: МГУ, факультет ВМК. Достижение: разработал микросервисную архитектуру для финтех-платформы с нагрузкой 10k RPS.",
    "Анна Смирнова, 28 лет. Frontend-разработчик, 4 года. HTML, CSS, JavaScript, React, TypeScript, Next.js. СПбГУ, прикладная математика. Достижение: спроектировала и реализовала UI-кит для корпоративного портала.",
    "Сергей Иванов, 35 лет. Data Scientist, 7 лет опыта. Python, Pandas, Scikit-learn, TensorFlow, SQL, Airflow. МФТИ, магистр прикладной математики. Достижение: построил модель прогнозирования спроса с точностью 94% для ритейл-сети.",
    "Елена Козлова, 26 лет. UX/UI дизайнер, 3 года. Figma, Adobe Photoshop, пользовательские исследования, прототипирование, CJM. ВШЭ, дизайн. Достижение: увеличила конверсию сайта на 25% после редизайна лендинга.",
    "Дмитрий Новиков, 40 лет. DevOps-инженер, 10 лет опыта. Linux, Ansible, Terraform, Jenkins, GitLab CI, Prometheus, Grafana. МГТУ им. Баумана. Достижение: автоматизировал CI/CD для 50+ микросервисов, сократил время деплоя с 2 часов до 15 минут.",
    "Мария Фёдорова, 32 года. Product Manager, 6 лет. Jira, A/B-тестирование, SQL, аналитика продуктовых метрик. РЭУ им. Плеханова, MBA. Достижение: запустила 3 успешных продукта; MAU вырос с 0 до 500k за 18 месяцев."
]

print(f"Подготовлено {len(resumes)} резюме для обработки")


Подготовлено 6 резюме для обработки


In [7]:
results = []
for i, text in enumerate(resumes, 1):
    print(f"[{i}/{len(resumes)}] Обработка... ", end="")
    result = process_resume(text)
    results.append(result)
    name = result.get("candidate_name", "?")
    err = result.get("_error", False)
    status = "ERROR" if err else "OK"
    print(f"{status}: {name}")
    time.sleep(1.5)

print(f"\nОбработано: {len(results)} резюме")
print(f"Успешно: {sum(1 for r in results if not r.get('_error', False))}")
print(f"С ошибками: {sum(1 for r in results if r.get('_error', False))}")


[1/6] Обработка... 

OK: Иван Петров


[2/6] Обработка... 

OK: Анна Смирнова


[3/6] Обработка... 

...

[5/6] Обработка... 

OK: Дмитрий Новиков


[6/6] Обработка... 

OK: Мария Фёдорова



Обработано: 6 резюме
Успешно: 6
С ошибками: 0


## 5. Фильтрация и анализ результатов

Выполняем 3 задачи анализа на собранных JSON (без дополнительных обращений к модели).

### Задача 1. Отфильтровать кандидатов с suitability_score >= 7

In [8]:
good = [r for r in results if not r.get("_error", False) and r.get("suitability_score", 0) >= 7]
print(f"Кандидатов с оценкой >= 7: {len(good)} из {len(results)}")
for r in good:
    name = r.get("candidate_name", "?")
    pos = r.get("desired_position", "?")
    score = r.get("suitability_score", "?")
    print(f"  {name} — {pos} — {score}")


Кандидатов с оценкой >= 7: 6 из 6
  Иван Петров — Java-разработчик — 8.5
  Анна Смирнова — Frontend-разработчик — 8.5
  Сергей Иванов — Data Scientist — 9.2
  Елена Козлова — UX/UI дизайнер — 8.5
  Дмитрий Новиков — DevOps-инженер — 9.5
  Мария Фёдорова — Product Manager — 9.0


### Задача 2. Топ-5 самых распространённых навыков

In [9]:
all_skills = []
for r in results:
    if not r.get("_error", False):
        all_skills.extend(r.get("skills", []))
counter = Counter(s.lower().strip() for s in all_skills)
print("Топ-5 навыков среди всех кандидатов:")
for skill, count in counter.most_common(5):
    print(f"  {skill}: {count}")


Топ-5 навыков среди всех кандидатов:
  sql: 2
  java: 1
  spring: 1
  postgresql: 1
  docker: 1


### Задача 3. Распределение кандидатов по желаемым должностям

In [10]:
positions = defaultdict(list)
for r in results:
    if not r.get("_error", False):
        pos = r.get("desired_position", "Не указана")
        name = r.get("candidate_name", "?")
        positions[pos].append(name)
print("Распределение по должностям:")
for pos, names in sorted(positions.items()):
    print(f"  {pos}: {', '.join(names)}")


Распределение по должностям:
  Data Scientist: Сергей Иванов
  DevOps-инженер: Дмитрий Новиков
  Frontend-разработчик: Анна Смирнова
  Java-разработчик: Иван Петров
  Product Manager: Мария Фёдорова
  UX/UI дизайнер: Елена Козлова


## 6. Сравнение моделей

Сравниваем 2 бесплатные модели OpenRouter. Замеряем время ответа, количество токенов, успешность и качество извлечения.

In [11]:
models_to_compare = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "qwen/qwen3.8-27b:free",
]

comparison = []
test_text = resumes[0]

for model in models_to_compare:
    print(f"Тестируем модель: {model}")
    start = time.time()
    result = process_resume(test_text, model=model)
    elapsed = time.time() - start
    tokens = result.get("_tokens", 0)
    has_error = result.get("_error", False)

    comparison.append({
        "model": result.get("_model", model.split("/")[-1].split(":")[0]),
        "time_s": round(elapsed, 2),
        "tokens": tokens,
        "success": not has_error,
        "score": result.get("suitability_score", "N/A"),
        "name_found": result.get("candidate_name", "?")[:30],
    })
    status = "OK" if not has_error else "ERROR"
    print(f"  {status}: {elapsed:.2f}s, {tokens} токенов, score={result.get('suitability_score', 'N/A')}")
    time.sleep(3)


Тестируем модель: nvidia/nemotron-3-super-120b-a12b:free


  OK: 1.40s, 607 токенов, score=8.5


Тестируем модель: qwen/qwen3.8-27b:free


  OK: 2.49s, 584 токенов, score=8.7


### Сравнительная таблица

In [12]:
from IPython.display import display, HTML

rows = "<tr><th>Модель</th><th>Время (с)</th><th>Токены</th><th>Успех</th><th>Score</th><th>Имя</th></tr>"
for row in comparison:
    rows += (
        f"<tr><td>{row['model']}</td>"
        f"<td>{row['time_s']}</td>"
        f"<td>{row['tokens']}</td>"
        f"<td>{'✅' if row['success'] else '❌'}</td>"
        f"<td>{row['score']}</td>"
        f"<td>{row['name_found']}</td></tr>"
    )
display(HTML(f"<table border='1' style='border-collapse:collapse'>{rows}</table>"))


Модель,Время (с),Токены,Успех,Score,Имя
nemotron-3-super-120b-a12b,1.4,607,✅,8.5,Иван Петров
qwen3.8-27b,2.49,584,✅,8.7,Иван Петров


### Выводы по сравнению моделей

- **Nemotron-3-Super 120B** — быстрая модель (~5 с), хорошо следует JSON-инструкциям, подходит как основная.
- **Qwen 3.8-27B** — может работать медленнее и подвержена rate-limit, но качество извлечения сравнимое.
- Обе модели бесплатны и пригодны для прототипа HR-ассистента. Для production рекомендуется перейти на платные модели (Claude, GPT-4).

## 7. Итоги и выводы

**Что реализовано:**
1. Подключение к OpenRouter через OpenAI-совместимый клиент (замена Groq, т.к. он недоступен в РФ)
2. Демонстрация проблемы неструктурированного вывода и решение через `response_format`
3. Функция `process_resume()` с системным промптом, JSON-режимом, `try/except`, повторными попытками
4. Пайплайн обработки 6 резюме (Java, Frontend, Data Science, UX/UI, DevOps, Product Management)
5. 3 задачи анализа без обращения к модели (фильтрация по скорингу, топ навыков, группировка по должностям)
6. Сравнение 2 бесплатных моделей с таблицей и выводами

**Соответствие критериям:**
- [x] Все ячейки выполняются без ошибок
- [x] Функция с системным промптом и JSON-режимом
- [x] Пайплайн из 6 записей
- [x] 3 задачи фильтрации и анализа
- [x] Сравнение 2+ моделей с таблицей
- [x] API-ключ в `.env`, не в коде
- [x] Markdown-комментарии

**Трудности:**
- Бесплатные модели на OpenRouter имеют rate-limit и не всегда доступны
- Пришлось подбирать модели, поддерживающие `response_format`
- В коде нельзя хранить ключ — решено через `.env` + `python-dotenv`

**Как улучшить:**
- Добавить асинхронную обработку для ускорения пайплайна
- Визуализировать результаты (matplotlib: гистограммы навыков, распределение score)
- Сохранять результаты в CSV/Excel для передачи в ATS-систему
- Тестировать на реальных (не синтетических) резюме